# HoloSyn Visual Interface (Gradio) — Integrates Your Distilled TorchScript Model

This Colab notebook builds a **visual UI** to:
- Load your distilled model: `/mnt/data/student_distilled_heads_hf.torchscript.pt`
- Load normalization: `/mnt/data/student_norm_hf.json`
- Optionally load your archive: `/mnt/data/Archive.zip` and browse files
- Compute modality-specific features (text/audio/image/video/haptics)
- Run inference → **valence/arousal/calm/trust**
- Visualize:
  - meters + time series
  - optional **two-peer synchrony** (A/B)
- Export a JSON session log

**Privacy note:** Everything runs locally in the notebook runtime.

---

## Inputs expected
- `/mnt/data/student_distilled_heads_hf.torchscript.pt`
- `/mnt/data/student_norm_hf.json`
- Optional: `/mnt/data/Archive.zip`


In [1]:
#@title 0) Install deps
#@title 0) Install deps
!pip -q install -U gradio numpy "pandas==2.2.2" pillow opencv-python soundfile librosa ffmpeg-python sentence-transformers transformers
!pip -q install -U torch torchvision torchaudio pyarrow
import os, json, time, zipfile
import numpy as np
import pandas as pd
from pathlib import Path
from PIL import Image

import torch
import librosa
import soundfile as sf
import cv2

import gradio as gr
print("✅ Installed")

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
fastai 2.8.6 requires torch<2.10,>=1.10, but you have torch 2.10.0 which is incompatible.
✅ Installed


In [2]:
#@title 1) Paths + load model
MODEL_PATH = "/content/student_distilled_heads_hf.torchscript.pt"
NORM_PATH  = "/content/student_norm_hf.json"
ARCHIVE_ZIP = "/content/Archive.zip"

assert os.path.exists(MODEL_PATH), f"Missing model: {MODEL_PATH}"
assert os.path.exists(NORM_PATH), f"Missing norm: {NORM_PATH}"

model = torch.jit.load(MODEL_PATH)
model.eval()

with open(NORM_PATH, "r", encoding="utf-8") as f:
    norm = json.load(f)

NUMERIC_COLS = norm["numeric_cols"]
MU = np.array(norm["mu"], dtype=np.float32)
SD = np.array(norm["sd"], dtype=np.float32)

print("✅ Model loaded")
print("Feature dims:", len(NUMERIC_COLS))

✅ Model loaded
Feature dims: 789


In [3]:
#@title 2) Optional: extract archive and index files
EXTRACT_DIR = "/content/archive_extracted_ui"
Path(EXTRACT_DIR).mkdir(parents=True, exist_ok=True)

AUDIO_EXT = {".wav",".mp3",".m4a",".flac",".ogg",".aac"}
VIDEO_EXT = {".mp4",".mov",".mkv",".webm",".avi"}
IMAGE_EXT = {".png",".jpg",".jpeg",".webp",".bmp"}
TEXT_EXT  = {".txt",".md",".json",".csv",".tsv"}

def extract_archive_if_present():
    if os.path.exists(ARCHIVE_ZIP):
        with zipfile.ZipFile(ARCHIVE_ZIP, "r") as z:
            z.extractall(EXTRACT_DIR)
        return True
    return False

def walk_files(root):
    out=[]
    for p in Path(root).rglob("*"):
        if p.is_file():
            out.append(str(p))
    return out

def bucket(path):
    ext = Path(path).suffix.lower()
    low = path.lower()
    if ext in AUDIO_EXT: return "audio"
    if ext in VIDEO_EXT: return "video"
    if ext in IMAGE_EXT: return "image"
    if ext in TEXT_EXT:
        if ext in {".json",".csv",".tsv"} and any(k in low for k in ["hapt","vib","pattern","tact","rumble"]):
            return "haptics"
        return "text"
    if any(k in low for k in ["hapt","vib","pattern","tact","rumble"]):
        return "haptics"
    return "other"

HAS_ARCHIVE = extract_archive_if_present()
INDEX = {"audio":[], "video":[], "image":[], "text":[], "haptics":[], "other":[]}
if HAS_ARCHIVE:
    for f in walk_files(EXTRACT_DIR):
        INDEX[bucket(f)].append(f)

print("Archive present:", HAS_ARCHIVE)
if HAS_ARCHIVE:
    for k in ["audio","video","image","text","haptics","other"]:
        print(k, len(INDEX[k]))


Archive present: True
audio 49
video 52
image 304
text 49
haptics 92
other 25


In [4]:
#@title 3) Feature extraction (must align with training feature schema)
def load_text(path, max_chars=12000):
    return open(path, "r", encoding="utf-8", errors="ignore").read()[:max_chars]

def load_haptics_any(path):
    ext = Path(path).suffix.lower()
    if ext == ".json":
        try:
            return json.load(open(path, "r", encoding="utf-8", errors="ignore"))
        except Exception:
            return {"raw": load_text(path)}
    if ext in {".csv",".tsv"}:
        sep = "," if ext==".csv" else "\t"
        try:
            df = pd.read_csv(path, sep=sep)
            return {"columns": list(df.columns), "head": df.head(200).to_dict(orient="list")}
        except Exception:
            return {"raw": load_text(path)}
    return {"raw": load_text(path)}

def featurize_text(s):
    return {
        "txt_len": float(len(s)),
        "txt_lines": float(s.count("\n")+1),
        "txt_exclaim": float(s.count("!")),
        "txt_question": float(s.count("?")),
        "txt_caps_ratio": float(sum(c.isupper() for c in s)/max(1,len(s))),
    }

def featurize_haptics(h):
    raw = json.dumps(h)[:20000].lower()
    return {
        "hapt_len": float(len(raw)),
        "hapt_has_intensity": 1.0 if "intensity" in raw else 0.0,
        "hapt_has_freq": 1.0 if ("hz" in raw or "freq" in raw) else 0.0,
    }

def audio_features(path, sr=16000, max_seconds=15):
    y, _ = librosa.load(path, sr=sr, mono=True, duration=max_seconds)
    if y.size < 10:
        return {"aud_rms":0.0,"aud_zcr":0.0,"aud_centroid":0.0,"aud_tempo":0.0}
    rms = float(np.mean(librosa.feature.rms(y=y)))
    zcr = float(np.mean(librosa.feature.zero_crossing_rate(y)))
    centroid = float(np.mean(librosa.feature.spectral_centroid(y=y, sr=sr)))
    onset_env = librosa.onset.onset_strength(y=y, sr=sr)
    tempo = float(librosa.beat.tempo(onset_envelope=onset_env, sr=sr)[0]) if onset_env.size else 0.0
    return {"aud_rms":rms,"aud_zcr":zcr,"aud_centroid":centroid,"aud_tempo":tempo}

def image_quick_stats(path):
    im = Image.open(path).convert("RGB")
    arr = np.asarray(im).astype(np.float32)/255.0
    mean = arr.mean(axis=(0,1))
    std = arr.std(axis=(0,1))
    return {
        "img_w": float(arr.shape[1]), "img_h": float(arr.shape[0]),
        "img_mean_r": float(mean[0]), "img_mean_g": float(mean[1]), "img_mean_b": float(mean[2]),
        "img_std_r": float(std[0]), "img_std_g": float(std[1]), "img_std_b": float(std[2]),
    }

def sample_video_frames(video_path, every_n_frames=45, max_frames=4, target_size=224):
    cap = cv2.VideoCapture(video_path)
    frames=[]
    idx=0
    while cap.isOpened() and len(frames) < max_frames:
        ret, frame = cap.read()
        if not ret: break
        if idx % every_n_frames == 0:
            frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            h,w = frame.shape[:2]
            if max(h,w) > target_size:
                scale = target_size / max(h,w)
                frame = cv2.resize(frame, (int(w*scale), int(h*scale)))
            frames.append(frame)
        idx += 1
    cap.release()
    return frames

def make_feature_vector(modality, path_or_text):
    # Return dict of features; missing features are 0.0.
    feats = {}
    preview = None

    if modality == "text":
        s = path_or_text if isinstance(path_or_text, str) and "\n" in path_or_text else load_text(path_or_text)
        feats.update(featurize_text(s))
        preview = s[:1000]
    elif modality == "haptics":
        h = load_haptics_any(path_or_text)
        feats.update(featurize_haptics(h))
        preview = json.dumps(h)[:1000]
    elif modality == "audio":
        feats.update(audio_features(path_or_text))
        preview = f"Audio file: {Path(path_or_text).name}"
    elif modality == "image":
        feats.update(image_quick_stats(path_or_text))
        preview = Image.open(path_or_text).convert("RGB")
    elif modality == "video":
        frames = sample_video_frames(path_or_text)
        feats["vid_n_frames"] = float(len(frames))
        preview = Image.fromarray(frames[0]).convert("RGB") if frames else None
    else:
        preview = f"Unsupported modality for: {path_or_text}"

    # Build full vector in the exact order expected by the student
    x = np.zeros((len(NUMERIC_COLS),), dtype=np.float32)
    for i, col in enumerate(NUMERIC_COLS):
        if col in feats:
            x[i] = np.float32(feats[col])
        else:
            # embeddings columns (clip_*, w2v_*) are not computed here; keep 0 unless you add embed models
            x[i] = 0.0
    return x, feats, preview

def predict_from_x(x):
    xn = (x - MU) / SD
    xt = torch.tensor(xn, dtype=torch.float32).unsqueeze(0)
    with torch.no_grad():
        y = model(xt).squeeze(0).cpu().numpy().astype(np.float32)
    # [valence, arousal, calm, trust]
    return y

In [5]:
#@title 4) Session logger utilities
SESSION_LOG = []

def log_step(label, modality, source, y, feats):
    rec = {
        "ts": time.time(),
        "label": label,
        "modality": modality,
        "source": source,
        "valence": float(y[0]),
        "arousal": float(y[1]),
        "calm": float(y[2]),
        "trust": float(y[3]),
        "features": {k: float(v) if isinstance(v,(int,float,np.floating)) else str(v) for k,v in feats.items()}
    }
    SESSION_LOG.append(rec)

def export_log():
    out = "/mnt/data/holosyn_session_log.json"
    with open(out, "w", encoding="utf-8") as f:
        json.dump(SESSION_LOG, f, indent=2)
    return out

## 5) Gradio UI

Two panels:
- **Single input inference** (pick modality + source)
- **Two-peer synchrony**: run A & B and compute cosine similarity on `[valence, arousal, calm, trust]`


In [6]:
#@title 5) Build UI
def list_options(modality):
    if not HAS_ARCHIVE:
        return []
    return [p.replace(EXTRACT_DIR + "/", "") for p in INDEX.get(modality, [])][:500]

def resolve_path(rel_path):
    if rel_path is None or rel_path == "":
        return None
    return os.path.join(EXTRACT_DIR, rel_path)

def infer_one(modality, rel_file, free_text):
    if modality == "text" and (free_text and free_text.strip()):
        x, feats, preview = make_feature_vector("text", free_text)
        y = predict_from_x(x)
        log_step("single", "text", "free_text", y, feats)
        return preview, float(y[0]), float(y[1]), float(y[2]), float(y[3]), feats
    else:
        path = resolve_path(rel_file)
        if not path or not os.path.exists(path):
            return "Pick a file from the dropdown (or paste text).", 0,0,0,0, {}
        x, feats, preview = make_feature_vector(modality, path)
        y = predict_from_x(x)
        log_step("single", modality, rel_file, y, feats)
        # preview might be PIL Image
        return preview, float(y[0]), float(y[1]), float(y[2]), float(y[3]), feats

def infer_pair(modalityA, fileA, textA, modalityB, fileB, textB):
    prevA, vA, aA, cA, tA, featsA = infer_one(modalityA, fileA, textA)
    prevB, vB, aB, cB, tB, featsB = infer_one(modalityB, fileB, textB)
    eA = np.array([vA,aA,cA,tA], dtype=np.float32)
    eB = np.array([vB,aB,cB,tB], dtype=np.float32)
    sync = float(np.dot(eA, eB) / (np.linalg.norm(eA)*np.linalg.norm(eB) + 1e-8))
    return prevA, prevB, sync

def do_export():
    path = export_log()
    return path

with gr.Blocks(title="HoloSyn UI") as demo:
    gr.Markdown("## HoloSyn Visual Interface (Local Distilled Model)")
    if not HAS_ARCHIVE:
        gr.Markdown("⚠️ `Archive.zip` not found. Upload it to `/mnt/data/Archive.zip` for file browsing. Text mode still works.")

    with gr.Tab("Single"):
        with gr.Row():
            modality = gr.Dropdown(choices=["text","audio","image","video","haptics"], value="text", label="Modality")
            file_dd = gr.Dropdown(choices=list_options("text"), label="File from Archive (optional)")
        free_text = gr.Textbox(lines=6, label="Text input (only used if modality=text and non-empty)")
        run_btn = gr.Button("Run inference")
        with gr.Row():
            preview = gr.Image(label="Preview (image/video frame) OR Text snippet", type="pil")
            preview_txt = gr.Textbox(label="Preview text (if not an image)", lines=8)
        with gr.Row():
            val = gr.Slider(0,1, step=0.001, label="Valence", interactive=False)
            aro = gr.Slider(0,1, step=0.001, label="Arousal", interactive=False)
            calm = gr.Slider(0,1, step=0.001, label="Calm", interactive=False)
            trust = gr.Slider(0,1, step=0.001, label="Trust", interactive=False)
        feats_json = gr.JSON(label="Extracted features used by student")

        def refresh_files(m):
            return gr.Dropdown.update(choices=list_options(m), value=None)
        modality.change(refresh_files, inputs=[modality], outputs=[file_dd])

        def render(preview_obj, v,a,c,t, feats):
            # If preview is an image, show it; else show text in preview_txt
            if isinstance(preview_obj, Image.Image):
                return preview_obj, "", v,a,c,t, feats
            else:
                # show placeholder image blank
                return None, str(preview_obj), v,a,c,t, feats

        run_btn.click(
            fn=lambda m,f,txt: infer_one(m,f,txt),
            inputs=[modality,file_dd,free_text],
            outputs=[preview_txt,val,aro,calm,trust,feats_json],
        ).then(
            fn=lambda prev_txt, v,a,c,t, feats: render(prev_txt, v,a,c,t, feats),
            inputs=[preview_txt,val,aro,calm,trust,feats_json],
            outputs=[preview,preview_txt,val,aro,calm,trust,feats_json]
        )

    with gr.Tab("Two-peer synchrony"):
        gr.Markdown("Run two inputs (A/B) and compute synchrony on the model outputs.")
        with gr.Row():
            modalityA = gr.Dropdown(choices=["text","audio","image","video","haptics"], value="text", label="Modality A")
            fileA = gr.Dropdown(choices=list_options("text"), label="File A")
        textA = gr.Textbox(lines=4, label="Text A (used if Modality A=text and non-empty)")
        with gr.Row():
            modalityB = gr.Dropdown(choices=["text","audio","image","video","haptics"], value="text", label="Modality B")
            fileB = gr.Dropdown(choices=list_options("text"), label="File B")
        textB = gr.Textbox(lines=4, label="Text B (used if Modality B=text and non-empty)")
        run_pair = gr.Button("Run pair + synchrony")
        with gr.Row():
            prevA = gr.Image(label="Preview A", type="pil")
            prevB = gr.Image(label="Preview B", type="pil")
        sync = gr.Slider(0,1, step=0.001, label="Synchrony (cosine similarity)", interactive=False)

        modalityA.change(lambda m: gr.Dropdown.update(choices=list_options(m), value=None), inputs=[modalityA], outputs=[fileA])
        modalityB.change(lambda m: gr.Dropdown.update(choices=list_options(m), value=None), inputs=[modalityB], outputs=[fileB])

        run_pair.click(
            fn=infer_pair,
            inputs=[modalityA,fileA,textA, modalityB,fileB,textB],
            outputs=[prevA,prevB,sync]
        )

    with gr.Tab("Export session log"):
        gr.Markdown("Exports all inference steps recorded so far to JSON.")
        export_btn = gr.Button("Export log")
        out_path = gr.Textbox(label="Saved to")
        export_btn.click(fn=do_export, inputs=[], outputs=[out_path])

demo.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://0402dc5b0467ccabfc.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [7]:
#@title 5) Build UI
def list_options(modality):
    if not HAS_ARCHIVE:
        return []
    return [p.replace(EXTRACT_DIR + "/", "") for p in INDEX.get(modality, [])][:500]

def resolve_path(rel_path):
    if rel_path is None or rel_path == "":
        return None
    return os.path.join(EXTRACT_DIR, rel_path)

def infer_one(modality, rel_file, free_text):
    if modality == "text" and (free_text and free_text.strip()):
        x, feats, preview = make_feature_vector("text", free_text)
        y = predict_from_x(x)
        log_step("single", "text", "free_text", y, feats)
        return preview, float(y[0]), float(y[1]), float(y[2]), float(y[3]), feats
    else:
        path = resolve_path(rel_file)
        if not path or not os.path.exists(path):
            return "Pick a file from the dropdown (or paste text).", 0,0,0,0, {}
        x, feats, preview = make_feature_vector(modality, path)
        y = predict_from_x(x)
        log_step("single", modality, rel_file, y, feats)
        return preview, float(y[0]), float(y[1]), float(y[2]), float(y[3]), feats

def do_export():
    path = export_log()
    return path

with gr.Blocks(title="HoloSyn UI") as demo:
    gr.Markdown("## HoloSyn Visual Interface (Local Distilled Model)")
    if not HAS_ARCHIVE:
        gr.Markdown("⚠️ `Archive.zip` not found. Upload it to `/mnt/data/Archive.zip` for file browsing. Text mode still works.")

    with gr.Tab("Single"):
        with gr.Row():
            modality = gr.Dropdown(choices=["text","audio","image","video","haptics"], value="text", label="Modality")
            file_dd = gr.Dropdown(choices=list_options("text"), label="File from Archive (optional)")
        free_text = gr.Textbox(lines=6, label="Text input (only used if modality=text and non-empty)")
        run_btn = gr.Button("Run inference")

        with gr.Row():
            preview = gr.Image(label="Preview (image/video frame)", type="pil")
            preview_txt = gr.Textbox(label="Preview text (if not an image)", lines=8)

        with gr.Row():
            val = gr.Slider(0,1, step=0.001, label="Valence", interactive=False)
            aro = gr.Slider(0,1, step=0.001, label="Arousal", interactive=False)
            calm = gr.Slider(0,1, step=0.001, label="Calm", interactive=False)
            trust = gr.Slider(0,1, step=0.001, label="Trust", interactive=False)
        feats_json = gr.JSON(label="Extracted features used by student")

        def refresh_files(m):
            return gr.Dropdown.update(choices=list_options(m), value=None)
        modality.change(refresh_files, inputs=[modality], outputs=[file_dd])

        # --- FIX: Unified handler to safely route text vs images ---
        def process_single(m, f, txt):
            prev_obj, v, a, c, t, feats = infer_one(m, f, txt)
            if isinstance(prev_obj, Image.Image):
                return prev_obj, "", v, a, c, t, feats
            else:
                return None, str(prev_obj), v, a, c, t, feats

        run_btn.click(
            fn=process_single,
            inputs=[modality, file_dd, free_text],
            outputs=[preview, preview_txt, val, aro, calm, trust, feats_json]
        )

    with gr.Tab("Two-peer synchrony"):
        gr.Markdown("Run two inputs (A/B) and compute synchrony on the model outputs.")
        with gr.Row():
            modalityA = gr.Dropdown(choices=["text","audio","image","video","haptics"], value="text", label="Modality A")
            fileA = gr.Dropdown(choices=list_options("text"), label="File A")
        textA = gr.Textbox(lines=4, label="Text A (used if Modality A=text and non-empty)")

        with gr.Row():
            modalityB = gr.Dropdown(choices=["text","audio","image","video","haptics"], value="text", label="Modality B")
            fileB = gr.Dropdown(choices=list_options("text"), label="File B")
        textB = gr.Textbox(lines=4, label="Text B (used if Modality B=text and non-empty)")

        run_pair = gr.Button("Run pair + synchrony")

        # --- FIX: Provide both Image and Text preview blocks for safely rendering dynamic modalities ---
        with gr.Row():
            with gr.Column():
                prevA_img = gr.Image(label="Preview A (Image/Video)", type="pil")
                prevA_txt = gr.Textbox(label="Preview A (Text/Other)", lines=4)
            with gr.Column():
                prevB_img = gr.Image(label="Preview B (Image/Video)", type="pil")
                prevB_txt = gr.Textbox(label="Preview B (Text/Other)", lines=4)

        sync = gr.Slider(0,1, step=0.001, label="Synchrony (cosine similarity)", interactive=False)

        modalityA.change(lambda m: gr.Dropdown.update(choices=list_options(m), value=None), inputs=[modalityA], outputs=[fileA])
        modalityB.change(lambda m: gr.Dropdown.update(choices=list_options(m), value=None), inputs=[modalityB], outputs=[fileB])

        # --- FIX: Unified handler to safely route pairs ---
        def process_pair(mA, fA, txtA, mB, fB, txtB):
            prevA, vA, aA, cA, tA, featsA = infer_one(mA, fA, txtA)
            prevB, vB, aB, cB, tB, featsB = infer_one(mB, fB, txtB)

            eA = np.array([vA,aA,cA,tA], dtype=np.float32)
            eB = np.array([vB,aB,cB,tB], dtype=np.float32)
            sync_val = float(np.dot(eA, eB) / (np.linalg.norm(eA)*np.linalg.norm(eB) + 1e-8))

            outA_img = prevA if isinstance(prevA, Image.Image) else None
            outA_txt = "" if isinstance(prevA, Image.Image) else str(prevA)
            outB_img = prevB if isinstance(prevB, Image.Image) else None
            outB_txt = "" if isinstance(prevB, Image.Image) else str(prevB)

            return outA_img, outA_txt, outB_img, outB_txt, sync_val

        run_pair.click(
            fn=process_pair,
            inputs=[modalityA, fileA, textA, modalityB, fileB, textB],
            outputs=[prevA_img, prevA_txt, prevB_img, prevB_txt, sync]
        )

    with gr.Tab("Export session log"):
        gr.Markdown("Exports all inference steps recorded so far to JSON.")
        export_btn = gr.Button("Export log")
        out_path = gr.Textbox(label="Saved to")
        export_btn.click(fn=do_export, inputs=[], outputs=[out_path])

demo.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://86aba2e2ba8da80a62.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [8]:
#@title 1) Imports & Load Model
import os, json, time, zipfile
import numpy as np
import pandas as pd
from pathlib import Path
from PIL import Image
import torch
import librosa
import soundfile as sf
import cv2
import gradio as gr
from sentence_transformers import SentenceTransformer

MODEL_PATH = "./student_distilled_heads_hf.torchscript.pt"
NORM_PATH  = "./student_norm_hf.json"
ARCHIVE_ZIP = "./Archive.zip"

print("Loading text embedding model (this may take a moment)...")
embedder = SentenceTransformer('all-mpnet-base-v2')

assert os.path.exists(MODEL_PATH), f"Missing model: {MODEL_PATH}"
assert os.path.exists(NORM_PATH), f"Missing norm: {NORM_PATH}"

model = torch.jit.load(MODEL_PATH)
model.eval()

with open(NORM_PATH, "r", encoding="utf-8") as f:
    norm = json.load(f)

NUMERIC_COLS = norm["numeric_cols"]
MU = np.array(norm["mu"], dtype=np.float32)
SD = np.array(norm["sd"], dtype=np.float32)

print("✅ Model and Norm loaded")
print("Feature dims:", len(NUMERIC_COLS))

Loading text embedding model (this may take a moment)...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

✅ Model and Norm loaded
Feature dims: 789


In [9]:
#@title 1) Imports & Load Model
import os, json, time, zipfile
import numpy as np
import pandas as pd
from pathlib import Path
from PIL import Image
import torch
import librosa
import soundfile as sf
import cv2
import gradio as gr
from sentence_transformers import SentenceTransformer
import transformers
import huggingface_hub

print(f"Transformers version: {transformers.__version__}")
print(f"Hugging Face Hub version: {huggingface_hub.__version__}")

MODEL_PATH = "./student_distilled_heads_hf.torchscript.pt"
NORM_PATH  = "./student_norm_hf.json"
ARCHIVE_ZIP = "./Archive.zip"

print("Loading text embedding model (this may take a moment)...")
embedder = SentenceTransformer('all-mpnet-base-v2')

assert os.path.exists(MODEL_PATH), f"Missing model: {MODEL_PATH}"
assert os.path.exists(NORM_PATH), f"Missing norm: {NORM_PATH}"

model = torch.jit.load(MODEL_PATH)
model.eval()

with open(NORM_PATH, "r", encoding="utf-8") as f:
    norm = json.load(f)

NUMERIC_COLS = norm["numeric_cols"]
MU = np.array(norm["mu"], dtype=np.float32)
SD = np.array(norm["sd"], dtype=np.float32)

print("✅ Model and Norm loaded")
print("Feature dims:", len(NUMERIC_COLS))

Loading text embedding model (this may take a moment)...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

MPNetModel LOAD REPORT from: sentence-transformers/all-mpnet-base-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


✅ Model and Norm loaded
Feature dims: 789


In [10]:
#@title 2) Optional: extract archive and index files
EXTRACT_DIR = "/content/archive_extracted_ui"
Path(EXTRACT_DIR).mkdir(parents=True, exist_ok=True)

AUDIO_EXT = {".wav",".mp3",".m4a",".flac",".ogg",".aac"}
VIDEO_EXT = {".mp4",".mov",".mkv",".webm",".avi"}
IMAGE_EXT = {".png",".jpg",".jpeg",".webp",".bmp"}
TEXT_EXT  = {".txt",".md",".json",".csv",".tsv"}

def extract_archive_if_present():
    if os.path.exists(ARCHIVE_ZIP):
        with zipfile.ZipFile(ARCHIVE_ZIP, "r") as z:
            z.extractall(EXTRACT_DIR)
        return True
    return False

def walk_files(root):
    out=[]
    for p in Path(root).rglob("*"):
        if p.is_file():
            out.append(str(p))
    return out

def bucket(path):
    ext = Path(path).suffix.lower()
    low = path.lower()
    if ext in AUDIO_EXT: return "audio"
    if ext in VIDEO_EXT: return "video"
    if ext in IMAGE_EXT: return "image"
    if ext in TEXT_EXT:
        if ext in {".json",".csv",".tsv"} and any(k in low for k in ["hapt","vib","pattern","tact","rumble"]):
            return "haptics"
        return "text"
    if any(k in low for k in ["hapt","vib","pattern","tact","rumble"]):
        return "haptics"
    return "other"

HAS_ARCHIVE = extract_archive_if_present()
INDEX = {"audio":[], "video":[], "image":[], "text":[], "haptics":[], "other":[]}
if HAS_ARCHIVE:
    for f in walk_files(EXTRACT_DIR):
        INDEX[bucket(f)].append(f)

print("Archive present:", HAS_ARCHIVE)
if HAS_ARCHIVE:
    for k in ["audio","video","image","text","haptics","other"]:
        print(k, len(INDEX[k]))

Archive present: True
audio 49
video 52
image 304
text 49
haptics 92
other 25


In [11]:
#@title 3) Feature extraction
def load_text(path, max_chars=12000):
    return open(path, "r", encoding="utf-8", errors="ignore").read()[:max_chars]

def load_haptics_any(path):
    ext = Path(path).suffix.lower()
    if ext == ".json":
        try:
            return json.load(open(path, "r", encoding="utf-8", errors="ignore"))
        except Exception:
            return {"raw": load_text(path)}
    if ext in {".csv",".tsv"}:
        sep = "," if ext==".csv" else "\t"
        try:
            df = pd.read_csv(path, sep=sep)
            return {"columns": list(df.columns), "head": df.head(200).to_dict(orient="list")}
        except Exception:
            return {"raw": load_text(path)}
    return {"raw": load_text(path)}

def featurize_text(s):
    return {
        "txt_len": float(len(s)),
        "txt_lines": float(s.count("\n")+1),
        "txt_exclaim": float(s.count("!")),
        "txt_question": float(s.count("?")),
        "txt_caps_ratio": float(sum(c.isupper() for c in s)/max(1,len(s))),
    }

def featurize_haptics(h):
    raw = json.dumps(h)[:20000].lower()
    return {
        "hapt_len": float(len(raw)),
        "hapt_has_intensity": 1.0 if "intensity" in raw else 0.0,
        "hapt_has_freq": 1.0 if ("hz" in raw or "freq" in raw) else 0.0,
    }

def audio_features(path, sr=16000, max_seconds=15):
    y, _ = librosa.load(path, sr=sr, mono=True, duration=max_seconds)
    if y.size < 10:
        return {"aud_rms":0.0,"aud_zcr":0.0,"aud_centroid":0.0,"aud_tempo":0.0}
    rms = float(np.mean(librosa.feature.rms(y=y)))
    zcr = float(np.mean(librosa.feature.zero_crossing_rate(y)))
    centroid = float(np.mean(librosa.feature.spectral_centroid(y=y, sr=sr)))
    onset_env = librosa.onset.onset_strength(y=y, sr=sr)
    tempo = float(librosa.beat.tempo(onset_envelope=onset_env, sr=sr)[0]) if onset_env.size else 0.0
    return {"aud_rms":rms,"aud_zcr":zcr,"aud_centroid":centroid,"aud_tempo":tempo}

def image_quick_stats(path):
    im = Image.open(path).convert("RGB")
    arr = np.asarray(im).astype(np.float32)/255.0
    mean = arr.mean(axis=(0,1))
    std = arr.std(axis=(0,1))
    return {
        "img_w": float(arr.shape[1]), "img_h": float(arr.shape[0]),
        "img_mean_r": float(mean[0]), "img_mean_g": float(mean[1]), "img_mean_b": float(mean[2]),
        "img_std_r": float(std[0]), "img_std_g": float(std[1]), "img_std_b": float(std[2]),
    }

def sample_video_frames(video_path, every_n_frames=45, max_frames=4, target_size=224):
    cap = cv2.VideoCapture(video_path)
    frames=[]
    idx=0
    while cap.isOpened() and len(frames) < max_frames:
        ret, frame = cap.read()
        if not ret: break
        if idx % every_n_frames == 0:
            frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            h,w = frame.shape[:2]
            if max(h,w) > target_size:
                scale = target_size / max(h,w)
                frame = cv2.resize(frame, (int(w*scale), int(h*scale)))
            frames.append(frame)
        idx += 1
    cap.release()
    return frames

def make_feature_vector(modality, path_or_text):
    feats = {}
    preview = None

    if modality == "text":
        s = path_or_text if isinstance(path_or_text, str) and "\n" in path_or_text else load_text(path_or_text)
        feats.update(featurize_text(s))
        # Compute the 768-d embedding
        emb = embedder.encode(s)
        for i in range(768):
            feats[f"w2v_{i}"] = float(emb[i])
        preview = s[:1000]

    elif modality == "haptics":
        h = load_haptics_any(path_or_text)
        feats.update(featurize_haptics(h))
        preview = json.dumps(h)[:1000]
    elif modality == "audio":
        feats.update(audio_features(path_or_text))
        preview = f"Audio file: {Path(path_or_text).name}"
    elif modality == "image":
        feats.update(image_quick_stats(path_or_text))
        preview = Image.open(path_or_text).convert("RGB")
    elif modality == "video":
        frames = sample_video_frames(path_or_text)
        feats["vid_n_frames"] = float(len(frames))
        preview = Image.fromarray(frames[0]).convert("RGB") if frames else None
    else:
        preview = f"Unsupported modality for: {path_or_text}"

    # Build full vector
    x = np.zeros((len(NUMERIC_COLS),), dtype=np.float32)
    for i, col in enumerate(NUMERIC_COLS):
        if col in feats:
            x[i] = np.float32(feats[col])
        else:
            x[i] = 0.0
    return x, feats, preview

def predict_from_x(x):
    xn = (x - MU) / SD
    xt = torch.tensor(xn, dtype=torch.float32).unsqueeze(0)
    with torch.no_grad():
        y = model(xt).squeeze(0).cpu().numpy().astype(np.float32)
    return y

In [12]:
#@title 4) Session logger utilities
SESSION_LOG = []

def log_step(label, modality, source, y, feats):
    rec = {
        "ts": time.time(),
        "label": label,
        "modality": modality,
        "source": source,
        "valence": float(y[0]),
        "arousal": float(y[1]),
        "calm": float(y[2]),
        "trust": float(y[3]),
        "features": {k: float(v) if isinstance(v,(int,float,np.floating)) else str(v) for k,v in feats.items()}
    }
    SESSION_LOG.append(rec)

def export_log():
    out = "/mnt/data/holosyn_session_log.json"
    with open(out, "w", encoding="utf-8") as f:
        json.dump(SESSION_LOG, f, indent=2)
    return out

In [14]:
#@title 5) Build UI
def list_options(modality):
    if not HAS_ARCHIVE:
        return []
    return [p.replace(EXTRACT_DIR + "/", "") for p in INDEX.get(modality, [])][:500]

def resolve_path(rel_path):
    if rel_path is None or rel_path == "":
        return None
    return os.path.join(EXTRACT_DIR, rel_path)

def infer_one(modality, rel_file, free_text):
    if modality == "text" and (free_text and free_text.strip()):
        x, feats, preview = make_feature_vector("text", free_text)
        y = predict_from_x(x)
        log_step("single", "text", "free_text", y, feats)
        return preview, float(y[0]), float(y[1]), float(y[2]), float(y[3]), feats
    else:
        path = resolve_path(rel_file)
        if not path or not os.path.exists(path):
            return "Pick a file from the dropdown (or paste text).", 0,0,0,0, {}
        x, feats, preview = make_feature_vector(modality, path)
        y = predict_from_x(x)
        log_step("single", modality, rel_file, y, feats)
        return preview, float(y[0]), float(y[1]), float(y[2]), float(y[3]), feats

def do_export():
    path = export_log()
    return path

with gr.Blocks(title="HoloSyn UI") as demo:
    gr.Markdown("## HoloSyn Visual Interface (Local Distilled Model)")
    if not HAS_ARCHIVE:
        gr.Markdown("⚠️ `Archive.zip` not found. Upload it to `/mnt/data/Archive.zip` for file browsing. Text mode still works.")

    with gr.Tab("Single"):
        with gr.Row():
            modality = gr.Dropdown(choices=["text","audio","image","video","haptics"], value="text", label="Modality")
            file_dd = gr.Dropdown(choices=list_options("text"), label="File from Archive (optional)")
        free_text = gr.Textbox(lines=6, label="Text input (only used if modality=text and non-empty)")
        run_btn = gr.Button("Run inference")

        with gr.Row():
            preview = gr.Image(label="Preview (image/video frame)", type="pil")
            preview_txt = gr.Textbox(label="Preview text (if not an image)", lines=8)

        with gr.Row():
            val = gr.Slider(0,1, step=0.001, label="Valence", interactive=False)
            aro = gr.Slider(0,1, step=0.001, label="Arousal", interactive=False)
            calm = gr.Slider(0,1, step=0.001, label="Calm", interactive=False)
            trust = gr.Slider(0,1, step=0.001, label="Trust", interactive=False)
        feats_json = gr.JSON(label="Extracted features used by student")

        def refresh_files(m):
            return gr.update(choices=list_options(m), value=None)

        modality.change(refresh_files, inputs=[modality], outputs=[file_dd])

        def process_single(m, f, txt):
            prev_obj, v, a, c, t, feats = infer_one(m, f, txt)
            if isinstance(prev_obj, Image.Image):
                return prev_obj, "", v, a, c, t, feats
            else:
                return None, str(prev_obj), v, a, c, t, feats

        run_btn.click(
            fn=process_single,
            inputs=[modality, file_dd, free_text],
            outputs=[preview, preview_txt, val, aro, calm, trust, feats_json]
        )

    with gr.Tab("Two-peer synchrony"):
        gr.Markdown("Run two inputs (A/B) and compute synchrony on the model outputs.")
        with gr.Row():
            modalityA = gr.Dropdown(choices=["text","audio","image","video","haptics"], value="text", label="Modality A")
            fileA = gr.Dropdown(choices=list_options("text"), label="File A")
        textA = gr.Textbox(lines=4, label="Text A (used if Modality A=text and non-empty)")

        with gr.Row():
            modalityB = gr.Dropdown(choices=["text","audio","image","video","haptics"], value="text", label="Modality B")
            fileB = gr.Dropdown(choices=list_options("text"), label="File B")
        textB = gr.Textbox(lines=4, label="Text B (used if Modality B=text and non-empty)")

        run_pair = gr.Button("Run pair + synchrony")

        with gr.Row():
            with gr.Column():
                prevA_img = gr.Image(label="Preview A (Image/Video)", type="pil")
                prevA_txt = gr.Textbox(label="Preview A (Text/Other)", lines=4)
            with gr.Column():
                prevB_img = gr.Image(label="Preview B (Image/Video)", type="pil")
                prevB_txt = gr.Textbox(label="Preview B (Text/Other)", lines=4)

        sync = gr.Slider(0,1, step=0.001, label="Synchrony (cosine similarity)", interactive=False)

        modalityA.change(lambda m: gr.update(choices=list_options(m), value=None), inputs=[modalityA], outputs=[fileA])
        modalityB.change(lambda m: gr.update(choices=list_options(m), value=None), inputs=[modalityB], outputs=[fileB])

        def process_pair(mA, fA, txtA, mB, fB, txtB):
            prevA, vA, aA, cA, tA, featsA = infer_one(mA, fA, txtA)
            prevB, vB, aB, cB, tB, featsB = infer_one(mB, fB, txtB)

            eA = np.array([vA,aA,cA,tA], dtype=np.float32)
            eB = np.array([vB,aB,cB,tB], dtype=np.float32)
            sync_val = float(np.dot(eA, eB) / (np.linalg.norm(eA)*np.linalg.norm(eB) + 1e-8))

            outA_img = prevA if isinstance(prevA, Image.Image) else None
            outA_txt = "" if isinstance(prevA, Image.Image) else str(prevA)
            outB_img = prevB if isinstance(prevB, Image.Image) else None
            outB_txt = "" if isinstance(prevB, Image.Image) else str(prevB)

            return outA_img, outA_txt, outB_img, outB_txt, sync_val

        run_pair.click(
            fn=process_pair,
            inputs=[modalityA, fileA, textA, modalityB, fileB, textB],
            outputs=[prevA_img, prevA_txt, prevB_img, prevB_txt, sync]
        )

    with gr.Tab("Export session log"):
        gr.Markdown("Exports all inference steps recorded so far to JSON.")
        export_btn = gr.Button("Export log")
        out_path = gr.Textbox(label="Saved to")
        export_btn.click(fn=do_export, inputs=[], outputs=[out_path])

demo.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://3a6eecdaf3f87e80c0.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [15]:
#@title 2) Optional: extract archive and index files
EXTRACT_DIR = "/content/archive_extracted_ui"
Path(EXTRACT_DIR).mkdir(parents=True, exist_ok=True)

AUDIO_EXT = {".wav",".mp3",".m4a",".flac",".ogg",".aac"}
VIDEO_EXT = {".mp4",".mov",".mkv",".webm",".avi"}
IMAGE_EXT = {".png",".jpg",".jpeg",".webp",".bmp"}
TEXT_EXT  = {".txt",".md",".json",".csv",".tsv"}

def extract_archive_if_present():
    if os.path.exists(ARCHIVE_ZIP):
        with zipfile.ZipFile(ARCHIVE_ZIP, "r") as z:
            z.extractall(EXTRACT_DIR)
        return True
    return False

def walk_files(root):
    out=[]
    for p in Path(root).rglob("*"):
        if p.is_file():
            out.append(str(p))
    return out

def bucket(path):
    ext = Path(path).suffix.lower()
    low = path.lower()
    if ext in AUDIO_EXT: return "audio"
    if ext in VIDEO_EXT: return "video"
    if ext in IMAGE_EXT: return "image"
    if ext in TEXT_EXT:
        if ext in {".json",".csv",".tsv"} and any(k in low for k in ["hapt","vib","pattern","tact","rumble"]):
            return "haptics"
        return "text"
    if any(k in low for k in ["hapt","vib","pattern","tact","rumble"]):
        return "haptics"
    return "other"

HAS_ARCHIVE = extract_archive_if_present()
INDEX = {"audio":[], "video":[], "image":[], "text":[], "haptics":[], "other":[]}
if HAS_ARCHIVE:
    for f in walk_files(EXTRACT_DIR):
        INDEX[bucket(f)].append(f)

print("Archive present:", HAS_ARCHIVE)
if HAS_ARCHIVE:
    for k in ["audio","video","image","text","haptics","other"]:
        print(k, len(INDEX[k]))

Archive present: True
audio 49
video 52
image 304
text 49
haptics 92
other 25


In [16]:
#@title 3) Feature extraction
def load_text(path, max_chars=12000):
    return open(path, "r", encoding="utf-8", errors="ignore").read()[:max_chars]

def load_haptics_any(path):
    ext = Path(path).suffix.lower()
    if ext == ".json":
        try:
            return json.load(open(path, "r", encoding="utf-8", errors="ignore"))
        except Exception:
            return {"raw": load_text(path)}
    if ext in {".csv",".tsv"}:
        sep = "," if ext==".csv" else "\t"
        try:
            df = pd.read_csv(path, sep=sep)
            return {"columns": list(df.columns), "head": df.head(200).to_dict(orient="list")}
        except Exception:
            return {"raw": load_text(path)}
    return {"raw": load_text(path)}

def featurize_text(s):
    return {
        "txt_len": float(len(s)),
        "txt_lines": float(s.count("\n")+1),
        "txt_exclaim": float(s.count("!")),
        "txt_question": float(s.count("?")),
        "txt_caps_ratio": float(sum(c.isupper() for c in s)/max(1,len(s))),
    }

def featurize_haptics(h):
    raw = json.dumps(h)[:20000].lower()
    return {
        "hapt_len": float(len(raw)),
        "hapt_has_intensity": 1.0 if "intensity" in raw else 0.0,
        "hapt_has_freq": 1.0 if ("hz" in raw or "freq" in raw) else 0.0,
    }

def audio_features(path, sr=16000, max_seconds=15):
    y, _ = librosa.load(path, sr=sr, mono=True, duration=max_seconds)
    if y.size < 10:
        return {"aud_rms":0.0,"aud_zcr":0.0,"aud_centroid":0.0,"aud_tempo":0.0}
    rms = float(np.mean(librosa.feature.rms(y=y)))
    zcr = float(np.mean(librosa.feature.zero_crossing_rate(y)))
    centroid = float(np.mean(librosa.feature.spectral_centroid(y=y, sr=sr)))
    onset_env = librosa.onset.onset_strength(y=y, sr=sr)
    tempo = float(librosa.beat.tempo(onset_envelope=onset_env, sr=sr)[0]) if onset_env.size else 0.0
    return {"aud_rms":rms,"aud_zcr":zcr,"aud_centroid":centroid,"aud_tempo":tempo}

def image_quick_stats(path):
    im = Image.open(path).convert("RGB")
    arr = np.asarray(im).astype(np.float32)/255.0
    mean = arr.mean(axis=(0,1))
    std = arr.std(axis=(0,1))
    return {
        "img_w": float(arr.shape[1]), "img_h": float(arr.shape[0]),
        "img_mean_r": float(mean[0]), "img_mean_g": float(mean[1]), "img_mean_b": float(mean[2]),
        "img_std_r": float(std[0]), "img_std_g": float(std[1]), "img_std_b": float(std[2]),
    }

def sample_video_frames(video_path, every_n_frames=45, max_frames=4, target_size=224):
    cap = cv2.VideoCapture(video_path)
    frames=[]
    idx=0
    while cap.isOpened() and len(frames) < max_frames:
        ret, frame = cap.read()
        if not ret: break
        if idx % every_n_frames == 0:
            frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            h,w = frame.shape[:2]
            if max(h,w) > target_size:
                scale = target_size / max(h,w)
                frame = cv2.resize(frame, (int(w*scale), int(h*scale)))
            frames.append(frame)
        idx += 1
    cap.release()
    return frames

def make_feature_vector(modality, path_or_text):
    feats = {}
    preview = None

    if modality == "text":
        s = path_or_text if isinstance(path_or_text, str) and "\n" in path_or_text else load_text(path_or_text)
        feats.update(featurize_text(s))
        # Compute the 768-d embedding
        emb = embedder.encode(s)
        for i in range(768):
            feats[f"w2v_{i}"] = float(emb[i])
        preview = s[:1000]

    elif modality == "haptics":
        h = load_haptics_any(path_or_text)
        feats.update(featurize_haptics(h))
        preview = json.dumps(h)[:1000]
    elif modality == "audio":
        feats.update(audio_features(path_or_text))
        preview = f"Audio file: {Path(path_or_text).name}"
    elif modality == "image":
        feats.update(image_quick_stats(path_or_text))
        preview = Image.open(path_or_text).convert("RGB")
    elif modality == "video":
        frames = sample_video_frames(path_or_text)
        feats["vid_n_frames"] = float(len(frames))
        preview = Image.fromarray(frames[0]).convert("RGB") if frames else None
    else:
        preview = f"Unsupported modality for: {path_or_text}"

    # Build full vector
    x = np.zeros((len(NUMERIC_COLS),), dtype=np.float32)
    for i, col in enumerate(NUMERIC_COLS):
        if col in feats:
            x[i] = np.float32(feats[col])
        else:
            x[i] = 0.0
    return x, feats, preview

def predict_from_x(x):
    xn = (x - MU) / SD
    xt = torch.tensor(xn, dtype=torch.float32).unsqueeze(0)
    with torch.no_grad():
        y = model(xt).squeeze(0).cpu().numpy().astype(np.float32)
    return y

In [17]:
#@title 4) Session logger utilities
SESSION_LOG = []

def log_step(label, modality, source, y, feats):
    rec = {
        "ts": time.time(),
        "label": label,
        "modality": modality,
        "source": source,
        "valence": float(y[0]),
        "arousal": float(y[1]),
        "calm": float(y[2]),
        "trust": float(y[3]),
        "features": {k: float(v) if isinstance(v,(int,float,np.floating)) else str(v) for k,v in feats.items()}
    }
    SESSION_LOG.append(rec)

def export_log():
    out = "/mnt/data/holosyn_session_log.json"
    with open(out, "w", encoding="utf-8") as f:
        json.dump(SESSION_LOG, f, indent=2)
    return out

In [18]:
#@title 5) Build UI
def list_options(modality):
    if not HAS_ARCHIVE:
        return []
    return [p.replace(EXTRACT_DIR + "/", "") for p in INDEX.get(modality, [])][:500]

def resolve_path(rel_path):
    if rel_path is None or rel_path == "":
        return None
    return os.path.join(EXTRACT_DIR, rel_path)

def infer_one(modality, rel_file, free_text):
    if modality == "text" and (free_text and free_text.strip()):
        x, feats, preview = make_feature_vector("text", free_text)
        y = predict_from_x(x)
        log_step("single", "text", "free_text", y, feats)
        return preview, float(y[0]), float(y[1]), float(y[2]), float(y[3]), feats
    else:
        path = resolve_path(rel_file)
        if not path or not os.path.exists(path):
            return "Pick a file from the dropdown (or paste text).", 0,0,0,0, {}
        x, feats, preview = make_feature_vector(modality, path)
        y = predict_from_x(x)
        log_step("single", modality, rel_file, y, feats)
        return preview, float(y[0]), float(y[1]), float(y[2]), float(y[3]), feats

def do_export():
    path = export_log()
    return path

with gr.Blocks(title="HoloSyn UI") as demo:
    gr.Markdown("## HoloSyn Visual Interface (Local Distilled Model)")
    if not HAS_ARCHIVE:
        gr.Markdown("⚠️ `Archive.zip` not found. Upload it to `/mnt/data/Archive.zip` for file browsing. Text mode still works.")

    with gr.Tab("Single"):
        with gr.Row():
            modality = gr.Dropdown(choices=["text","audio","image","video","haptics"], value="text", label="Modality")
            file_dd = gr.Dropdown(choices=list_options("text"), label="File from Archive (optional)")
        free_text = gr.Textbox(lines=6, label="Text input (only used if modality=text and non-empty)")
        run_btn = gr.Button("Run inference")

        with gr.Row():
            preview = gr.Image(label="Preview (image/video frame)", type="pil")
            preview_txt = gr.Textbox(label="Preview text (if not an image)", lines=8)

        with gr.Row():
            val = gr.Slider(0,1, step=0.001, label="Valence", interactive=False)
            aro = gr.Slider(0,1, step=0.001, label="Arousal", interactive=False)
            calm = gr.Slider(0,1, step=0.001, label="Calm", interactive=False)
            trust = gr.Slider(0,1, step=0.001, label="Trust", interactive=False)
        feats_json = gr.JSON(label="Extracted features used by student")

        def refresh_files(m):
            return gr.update(choices=list_options(m), value=None)

        modality.change(refresh_files, inputs=[modality], outputs=[file_dd])

        def process_single(m, f, txt):
            prev_obj, v, a, c, t, feats = infer_one(m, f, txt)
            if isinstance(prev_obj, Image.Image):
                return prev_obj, "", v, a, c, t, feats
            else:
                return None, str(prev_obj), v, a, c, t, feats

        run_btn.click(
            fn=process_single,
            inputs=[modality, file_dd, free_text],
            outputs=[preview, preview_txt, val, aro, calm, trust, feats_json]
        )

    with gr.Tab("Two-peer synchrony"):
        gr.Markdown("Run two inputs (A/B) and compute synchrony on the model outputs.")
        with gr.Row():
            modalityA = gr.Dropdown(choices=["text","audio","image","video","haptics"], value="text", label="Modality A")
            fileA = gr.Dropdown(choices=list_options("text"), label="File A")
        textA = gr.Textbox(lines=4, label="Text A (used if Modality A=text and non-empty)")

        with gr.Row():
            modalityB = gr.Dropdown(choices=["text","audio","image","video","haptics"], value="text", label="Modality B")
            fileB = gr.Dropdown(choices=list_options("text"), label="File B")
        textB = gr.Textbox(lines=4, label="Text B (used if Modality B=text and non-empty)")

        run_pair = gr.Button("Run pair + synchrony")

        with gr.Row():
            with gr.Column():
                prevA_img = gr.Image(label="Preview A (Image/Video)", type="pil")
                prevA_txt = gr.Textbox(label="Preview A (Text/Other)", lines=4)
            with gr.Column():
                prevB_img = gr.Image(label="Preview B (Image/Video)", type="pil")
                prevB_txt = gr.Textbox(label="Preview B (Text/Other)", lines=4)

        sync = gr.Slider(0,1, step=0.001, label="Synchrony (cosine similarity)", interactive=False)

        modalityA.change(lambda m: gr.update(choices=list_options(m), value=None), inputs=[modalityA], outputs=[fileA])
        modalityB.change(lambda m: gr.update(choices=list_options(m), value=None), inputs=[modalityB], outputs=[fileB])

        def process_pair(mA, fA, txtA, mB, fB, txtB):
            prevA, vA, aA, cA, tA, featsA = infer_one(mA, fA, txtA)
            prevB, vB, aB, cB, tB, featsB = infer_one(mB, fB, txtB)

            eA = np.array([vA,aA,cA,tA], dtype=np.float32)
            eB = np.array([vB,aB,cB,tB], dtype=np.float32)
            sync_val = float(np.dot(eA, eB) / (np.linalg.norm(eA)*np.linalg.norm(eB) + 1e-8))

            outA_img = prevA if isinstance(prevA, Image.Image) else None
            outA_txt = "" if isinstance(prevA, Image.Image) else str(prevA)
            outB_img = prevB if isinstance(prevB, Image.Image) else None
            outB_txt = "" if isinstance(prevB, Image.Image) else str(prevB)

            return outA_img, outA_txt, outB_img, outB_txt, sync_val

        run_pair.click(
            fn=process_pair,
            inputs=[modalityA, fileA, textA, modalityB, fileB, textB],
            outputs=[prevA_img, prevA_txt, prevB_img, prevB_txt, sync]
        )

    with gr.Tab("Export session log"):
        gr.Markdown("Exports all inference steps recorded so far to JSON.")
        export_btn = gr.Button("Export log")
        out_path = gr.Textbox(label="Saved to")
        export_btn.click(fn=do_export, inputs=[], outputs=[out_path])

demo.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://3534f49ad8a464a888.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [21]:
!pip install cirq qsimcirq torch torchvision numpy
import torch
import torch.nn as nn
import torch.nn.functional as F
import cirq
import qsimcirq
import json
import numpy as np

In [22]:
# Load your multimodal normalization statistics
with open('student_norm_hf.json', 'r') as f:
    norm_stats = json.load(f)

# Extract means and standard deviations for PyTorch preprocessing
mu = torch.tensor(norm_stats['mu'], dtype=torch.float32)
sd = torch.tensor(norm_stats['sd'], dtype=torch.float32)

def normalize_features(features):
    # Avoid division by zero for any zero-variance features
    return (features - mu) / (sd + 1e-7)

In [23]:
class QSimLayer(nn.Module):
    def __init__(self, num_qubits):
        super(QSimLayer, self).__init__()
        self.num_qubits = num_qubits
        self.qubits = cirq.GridQubit.rect(1, num_qubits)
        self.simulator = qsimcirq.QSimSimulator()

        # Trainable PyTorch parameters for the quantum circuit
        self.theta = nn.Parameter(torch.rand(num_qubits) * np.pi)

    def forward(self, x):
        batch_size = x.size(0)
        out = torch.zeros(batch_size, self.num_qubits, device=x.device)

        # Iterating through the batch to simulate the quantum state
        for i in range(batch_size):
            circuit = cirq.Circuit()

            # 1. Feature Encoding: Mapping continuous variables (e.g., w2v) to Rx gates
            for j in range(self.num_qubits):
                feature_val = x[i, j].item()
                circuit.append(cirq.rx(feature_val)(self.qubits[j]))

            # 2. Trainable Variational Entanglement Layer
            for j in range(self.num_qubits - 1):
                circuit.append(cirq.CNOT(self.qubits[j], self.qubits[j+1]))
            for j in range(self.num_qubits):
                circuit.append(cirq.ry(self.theta[j].item())(self.qubits[j]))

            # 3. Simulate with qsimcirq for state-vector processing
            result = self.simulator.simulate(circuit)
            state_vector = result.state_vector()

            # Extract probabilities and project back to a real tensor
            probs = np.abs(state_vector)**2
            out[i] = torch.tensor(probs[:self.num_qubits], dtype=torch.float32)

        return out

In [24]:
# 1. Load the Teacher Model (e.g., your open source model or holosyn_heads)
teacher_model = torch.jit.load('holosyn_heads.torchscript.pt')
teacher_model.eval() # Freeze the teacher

# 2. Load the Student Model
student_model = torch.jit.load('student_distilled_heads_hf.torchscript.pt')
student_model.train()

# 3. Wrap student with the Quantum Layer
class QuantumDistilledStudent(nn.Module):
    def __init__(self, base_student, num_q_features=8):
        super(QuantumDistilledStudent, self).__init__()
        self.num_q_features = num_q_features
        # Routing a subset of features (like w2v embeddings) through the quantum layer
        self.quantum_layer = QSimLayer(num_qubits=num_q_features)
        self.base_student = base_student

    def forward(self, x):
        # Pass the first 'n' features through the VQC
        q_out = self.quantum_layer(x[:, :self.num_q_features])

        # Re-inject quantum features into the main tensor
        x_modified = x.clone()
        x_modified[:, :self.num_q_features] = q_out

        return self.base_student(x_modified)

model = QuantumDistilledStudent(student_model)

In [25]:
def distillation_loss(student_logits, teacher_logits, labels, T=2.0, alpha=0.5):
    """
    Combines KL Divergence from the teacher and Standard Loss from the labels.
    """
    hard_loss = F.cross_entropy(student_logits, labels)
    soft_loss = F.kl_div(
        F.log_softmax(student_logits / T, dim=1),
        F.softmax(teacher_logits / T, dim=1),
        reduction='batchmean'
    ) * (T * T)

    return alpha * hard_loss + (1. - alpha) * soft_loss

optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

# Pseudo-training execution
# for batch in dataloader:
#     features, labels = batch
#     features = normalize_features(features)
#
#     with torch.no_grad():
#         teacher_preds = teacher_model(features)
#
#     student_preds = model(features)
#     loss = distillation_loss(student_preds, teacher_preds, labels)
#
#     optimizer.zero_grad()
#     loss.backward()
#     optimizer.step()